# Validating and round-tripping `.eosmat`

The `.eosmat` format keeps a material's identity, optional crystallography,
EOS records, provenance, validity, and uncertainty metadata in one versioned
document. This notebook validates and round-trips a bundled material, then
creates a deliberately unaudited synthetic example to show the safety boundary
between valid data exchange and trusted scientific execution.

In [1]:
from copy import deepcopy
from pathlib import Path
from tempfile import TemporaryDirectory

from peritheos import (
    Material,
    eosmat_schema,
    get_material_document,
    load_eosmat,
    save_eosmat,
    validate_eosmat_document,
)

schema = eosmat_schema()
print("Schema identifier:", schema["$id"])
print("Required document fields:", ", ".join(schema["required"]))
print("Supported top-level fields:", ", ".join(schema["properties"]))

Schema identifier: https://peritheos.readthedocs.io/schemas/eosmat-v3.schema.json
Required document fields: format, format_version, identifier, name, formula, units, eos_records
Supported top-level fields: format, format_version, identifier, name, formula, phase, cell_contents, units, aliases, symmetry, lattice, formula_units_per_cell, space_group, space_group_number, atom_sites, source, notes, peaks, eos_records


## Round-trip a bundled validated document

The reader and writer preserve optional structure and record fields. A
temporary directory keeps this executable example from leaving generated
files in the source tree.

In [2]:
gold = get_material_document("gold")
validate_eosmat_document(gold)

with TemporaryDirectory() as directory:
    path = Path(directory) / "gold.eosmat"
    save_eosmat(path, gold)
    reloaded_gold = load_eosmat(path)
    print(f"Wrote and reloaded {path.name} ({path.stat().st_size} bytes)")

print("Exact dictionary round-trip:", reloaded_gold == gold)
for field in ("lattice", "space_group", "atom_sites", "eos_records"):
    print(f"Preserved {field}: {reloaded_gold[field] == gold[field]}")

Wrote and reloaded gold.eosmat (15809 bytes)
Exact dictionary round-trip: True
Preserved lattice: True
Preserved space_group: True
Preserved atom_sites: True
Preserved eos_records: True


## Create a minimal synthetic material document

Structural validity is not the same as primary-source validation. This record
uses plausible BM3 values only to demonstrate the format, so its scientific
status is explicitly `pending_primary_source_check` and its reference says
that the parameters are synthetic.

In [3]:
synthetic = {
    "format": "peritheos.material",
    "format_version": 3,
    "identifier": "synthetic_bm3_demo",
    "name": "Synthetic BM3 demonstration",
    "formula": "X",
    "units": {
        "pressure": "GPa",
        "temperature": "K",
        "volume": "angstrom^3/conventional_unit_cell",
    },
    "eos_records": [
        {
            "identifier": "synthetic_bm3_demo_1",
            "label": "Synthetic BM3 parameters",
            "reference": "Synthetic parameters for API demonstration only",
            "eos": {
                "type": "BM3",
                "model": "birch_murnaghan_3",
                "parameters": {"V0": 10.0, "K0": 150.0, "K0_prime": 4.5},
            },
            "parameter_errors": {
                "V0": None,
                "K0": None,
                "K0_prime": None,
            },
            "fixed_parameters": [],
            "temperature_ref": 300.0,
            "scientific_validation": {
                "status": "pending_primary_source_check",
                "note": "Synthetic demonstration; not a literature record.",
            },
        }
    ],
}

validate_eosmat_document(synthetic)
with TemporaryDirectory() as directory:
    path = Path(directory) / "synthetic.eosmat"
    save_eosmat(path, synthetic)
    reloaded_synthetic = load_eosmat(path)

print("Synthetic document is structurally valid:", reloaded_synthetic == synthetic)
print(
    "Scientific status:",
    reloaded_synthetic["eos_records"][0]["scientific_validation"]["status"],
)

Synthetic document is structurally valid: True
Scientific status: pending_primary_source_check


## Observe the execution safety boundary

The default constructor refuses the pending record. An explicit opt-in exists
for inspecting legacy or provisional data, but using it does not promote the
record or make its parameters suitable for scientific reporting.

In [4]:
try:
    Material.from_eosmat(reloaded_synthetic)
except ValueError as error:
    print("Default construction rejected the record:")
    print(" ", error)

provisional = Material.from_eosmat(
    reloaded_synthetic,
    require_primary_validation=False,
)
demo_pressure = provisional.eos_records[0].pressure(9.0)
print(f"Explicit provisional calculation at V=9.0: {demo_pressure:.3f} GPa")

Default construction rejected the record:
  Invalid EOS record 'synthetic_bm3_demo_1': record 'synthetic_bm3_demo_1' is 'pending_primary_source_check'; pass require_primary_validation=False only to accept unaudited parameters explicitly
Explicit provisional calculation at V=9.0: 20.048 GPa


## Invalid format versions fail early

Schema and compatibility checks prevent a consumer from silently interpreting
an unknown future format as version 3.

In [5]:
invalid = deepcopy(synthetic)
invalid["format_version"] = 99

try:
    validate_eosmat_document(invalid)
except ValueError as error:
    print(type(error).__name__ + ":", error)

ValueError: Supported eosmat documents are Peritheos format 3 or legacy Dioptas format 2


## Takeaways

1. Validate before writing or constructing executable models.
2. Round trips preserve EOS and optional crystallographic information.
3. Structural validity does not imply primary-source scientific validation.
4. Unknown format versions and unaudited records require explicit handling;
   neither is silently coerced into a trusted calculation.